# Relative effects and MDE for ratio metrics

For ratio outcomes (target/scale, e.g. CTR), **relative** inference uses an outer delta method: the variance of (ATE / control mean) accounts for uncertainty in both the numerator and the denominator.

- Implementation detail: formulas live in `cluster_experiments.relative_lift_transformer`; `DeltaMethodAnalysis` applies them for analysis.
- **Relative MDE** for power uses the double-delta (quadratic) formulation. Call it via **`DeltaMethodAnalysis.relative_ratio_mde(alpha, power, ctrl_mean, ctrl_var, treat_var)`** — the same path `NormalPowerAnalysis` uses internally (not `absolute_mde / control_mean`).

Rolling MDE timelines for delta + relative require **`agg_func='sum'`** per cluster (numerator and denominator totals).

In [1]:
import random

import numpy as np
import pandas as pd

from cluster_experiments.experiment_analysis import DeltaMethodAnalysis
from cluster_experiments.power_analysis import NormalPowerAnalysis
from cluster_experiments.random_splitter import ClusteredSplitter

import warnings

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)

## 1. Simulated user-level data
Each row: a user-session with `target` (e.g. clicks) and `scale` (e.g. impressions). Treatment is assigned at user level.

In [2]:
np.random.seed(42)
n_users = 2000
n_rows = 10000

df = pd.DataFrame({
    "user": np.random.choice([f"User {i}" for i in range(n_users)], size=n_rows),
    "scale": np.random.poisson(lam=10, size=n_rows).clip(1),
})

user_effects = {f"User {i}": np.random.normal(0, 0.1) for i in range(n_users)}
df["user_effect"] = df["user"].map(user_effects)
df["baseline_prob"] = (0.05 + df["user_effect"]).clip(0.01, 0.99)

splitter = ClusteredSplitter(cluster_cols=["user"])
df = splitter.assign_treatment_df(df)
df["is_treatment"] = (df["treatment"] == "B").astype(int)

TRUE_RELATIVE_LIFT = 0.15
df["prob"] = df["baseline_prob"] * (1 + TRUE_RELATIVE_LIFT * df["is_treatment"])
df["target"] = np.random.binomial(df["scale"], df["prob"])

# Dates for mde_time_line / mde_rolling_time_line
df["date"] = np.random.choice(pd.date_range("2024-01-01", periods=28), size=n_rows)

df.head()

,user,scale,user_effect,baseline_prob,treatment,is_treatment,prob,target,date
0,User 1126,10,-0.037478,0.012522,A,0,0.012522,1,2024-01-25
1,User 1459,13,0.030119,0.080119,B,1,0.092137,0,2024-01-20
2,User 860,15,0.012006,0.062006,B,1,0.071307,1,2024-01-02
3,User 1294,10,-0.088254,0.010000,A,0,0.010000,0,2024-01-09
4,User 1130,8,0.076286,0.126286,B,1,0.145229,1,2024-01-10


## 2. Naive relative vs proper delta (SE)
Point estimate of relative lift matches ATE/control mean; the **proper relative SE** is at least as large as the naive SE/|control mean|.

In [3]:
analyser_abs = DeltaMethodAnalysis(
    cluster_cols=["user"], scale_col="scale", target_col="target", relative_effect=False
)
analyser_rel = DeltaMethodAnalysis(
    cluster_cols=["user"], scale_col="scale", target_col="target", relative_effect=True
)

abs_ate = analyser_abs.get_point_estimate(df)
abs_se = analyser_abs.get_standard_error(df)
rel_ate = analyser_rel.get_point_estimate(df)
rel_se = analyser_rel.get_standard_error(df)

ctrl_df = df[df["treatment"] == "A"]
ctrl_mean = ctrl_df["target"].sum() / ctrl_df["scale"].sum()

naive_rel_ate = abs_ate / ctrl_mean
naive_rel_se = abs_se / ctrl_mean

print(f"Control ratio mean: {ctrl_mean:.4f}")
print(f"Absolute ATE / SE:  {abs_ate:.4f} / {abs_se:.4f}")
print(f"Naive rel ATE/SE:   {naive_rel_ate:.4f} / {naive_rel_se:.4f}")
print(f"Proper rel ATE/SE:  {rel_ate:.4f} / {rel_se:.4f}")
print(f"Proper rel SE >= naive rel SE: {rel_se >= naive_rel_se - 1e-9}")

Control ratio mean: 0.0752
Absolute ATE / SE:  0.0106 / 0.0042
Naive rel ATE/SE:   0.1404 / 0.0557
Proper rel ATE/SE:  0.1404 / 0.0592
Proper rel SE >= naive rel SE: True


## 3. Absolute MDE vs relative MDE (quadratic)
`pw_rel.mde(...)` uses the double-delta relative MDE. With the **same RNG seed** before averaging simulated `(ctrl_mean, ctrl_var, treat_var)`, **`DeltaMethodAnalysis.relative_ratio_mde`** matches `mde_power_line` exactly.

In [4]:
pw_abs = NormalPowerAnalysis(
    splitter=splitter, analysis=analyser_abs, n_simulations=50, seed=1
)
pw_rel = NormalPowerAnalysis(
    splitter=splitter, analysis=analyser_rel, n_simulations=50, seed=1
)

mde_abs = pw_abs.mde(df, power=0.8)
mde_rel = pw_rel.mde(df, power=0.8)
naive_mde_rel = mde_abs / ctrl_mean

np.random.seed(99)
random.seed(99)
cm, cv, tv = pw_rel._get_average_ratio_mde_stats(df, n_simulations=50)
mde_from_facade = DeltaMethodAnalysis.relative_ratio_mde(0.05, 0.8, cm, cv, tv)
np.random.seed(99)
random.seed(99)
mde_from_power_line = pw_rel.mde_power_line(df, powers=[0.8], n_simulations=50)[0.8]

print(f"Absolute MDE:              {mde_abs:.4f}")
print(f"Naive relative MDE:        {naive_mde_rel:.4f}  (mde_abs / ctrl_mean — optimistic)")
print(f"Proper relative MDE (pw):  {mde_rel:.4f}")
print(f"Facade == mde_power_line:  {mde_from_facade:.6f} == {mde_from_power_line:.6f}")

Absolute MDE:              0.0117
Naive relative MDE:        0.1556  (mde_abs / ctrl_mean — optimistic)
Proper relative MDE (pw):  0.1487
Facade == mde_power_line:  0.148662 == 0.148662


## 4. MDE vs experiment length (time line)
With `DeltaMethodAnalysis` + `relative_effect=True`, `mde_time_line` returns **dimensionless relative MDE** per window (quadratic), not absolute MDE.

In [5]:
df["date"] = pd.to_datetime(df["date"])
pw_timeline = NormalPowerAnalysis(
    splitter=splitter,
    analysis=analyser_rel,
    n_simulations=30,
    time_col="date",
    seed=2,
)
line = pw_timeline.mde_time_line(
    df, experiment_length=[7, 14, 21], powers=[0.8], n_simulations=30
)
pd.DataFrame(line)

,power,mde,experiment_length
0,0.8,0.213889,7
1,0.8,0.172554,14
2,0.8,0.156322,21


## 5. Rolling cumulative window (delta + relative)
Use **`agg_func='sum'`** so each cluster has total target and scale in the window. `mde` and `relative_mde` are the same (relative MDE).

In [6]:
rolling = pw_timeline.mde_rolling_time_line(
    df,
    experiment_length=[7, 14],
    powers=[0.8],
    n_simulations=25,
    agg_func="sum",
)
pd.DataFrame(rolling)

,power,mde,experiment_length,relative_mde
0,0.8,0.201887,7,0.201887
1,0.8,0.165671,14,0.165671
